# CS-4795 AI Term Project
### 3x3 Rubik's Cube Solver : Bidirectional BFS
### By: Fady Elgohary (3762991) & Darin Thomson (3776723)

**Algorithm: Bidirectional BFS**

IDA* with a weak heuristic blows up exponentially at depth 10+.
Bidirectional BFS searches from **both ends simultaneously**: from the scrambled state forward, and from the solved state backward and meets in the middle.

- One-directional BFS at depth 10: explores ~18^10 ≈ 357 billion nodes
- Bidirectional BFS at depth 10: explores ~2 × 18^5 ≈ 3.5 million nodes: **100,000× fewer**
- Solves any standard scramble in **seconds**

---
## 1. Imports

In [1]:
import sys
import time
from collections import Counter, deque

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False
    print("cv2 not available — image loading disabled, solver still works.")

---
## 2. Color Detection from Images

In [2]:
def detect_color(hsv_pixel):
    """Map a single HSV pixel to one of 6 cube colors."""
    h, s, v = hsv_pixel
    if s < 60 and v > 160:
        return 'W'
    if h < 7 or h > 170: return 'R'
    if h < 22:           return 'O'
    if h < 45:           return 'Y'
    if h < 95:           return 'G'
    if h < 150:          return 'B'
    return 'W'


def process_face(image_path):
    if not CV2_AVAILABLE:
        print(f"WARNING: cv2 not available. Using all-White face for {image_path}.")
        return ['W'] * 9

    img = cv2.imread(image_path)
    if img is None:
        print(f"WARNING: Could not load {image_path}. Using all-White face.")
        return ['W'] * 9

    img  = cv2.resize(img, (300, 300))
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    face = []
    cell = 100
    m    = 15

    for row in range(3):
        for col in range(3):
            roi    = hsv[row*cell+m:(row+1)*cell-m, col*cell+m:(col+1)*cell-m]
            pixels = roi.reshape(-1, 3)
            counts = {'W':0,'R':0,'G':0,'Y':0,'O':0,'B':0}
            for p in pixels:
                counts[detect_color(p)] += 1
            face.append(max(counts, key=counts.get))

    print(f"{image_path}: {face}")
    return face

---
## 3. Build Cube from Images

In [3]:
image_paths = ['U.png', 'D.png', 'F.png', 'B.png', 'L.png', 'R.png']

def build_cube_from_images():
    stickers = []
    for path in image_paths:
        stickers.extend(process_face(path))
    return tuple(stickers)

---
## 4. Cube Representation & Move Engine

In [4]:
# 54-char tuple: 9×W, 9×Y, 9×G, 9×B, 9×O, 9×R
SOLVED = tuple('W'*9 + 'Y'*9 + 'G'*9 + 'B'*9 + 'O'*9 + 'R'*9)

U, D, F, B, L, R = 0, 9, 18, 27, 36, 45

def is_solved(cube):
    return cube == SOLVED


def cycle_perm(*cycles):
    perm = list(range(54))
    for cyc in cycles:
        n = len(cyc)
        for i in range(n):
            perm[cyc[i]] = cyc[(i + 1) % n]
    return tuple(perm)

def compose(p1, p2):
    return tuple(p1[p2[i]] for i in range(54))

def invert_perm(p):
    inv = [0] * 54
    for i, v in enumerate(p):
        inv[v] = i
    return tuple(inv)

def face_cw_cycles(base):
    b = base
    return [
        (b+0, b+6, b+8, b+2),
        (b+1, b+3, b+7, b+5),
    ]

_U = cycle_perm(*face_cw_cycles(U), (F+0,L+0,B+0,R+0), (F+1,L+1,B+1,R+1), (F+2,L+2,B+2,R+2))
_D = cycle_perm(*face_cw_cycles(D), (F+6,R+6,B+6,L+6), (F+7,R+7,B+7,L+7), (F+8,R+8,B+8,L+8))
_F = cycle_perm(*face_cw_cycles(F), (U+6,R+0,D+2,L+8), (U+7,R+3,D+1,L+5), (U+8,R+6,D+0,L+2))
_B = cycle_perm(*face_cw_cycles(B), (U+2,L+0,D+6,R+8), (U+1,L+3,D+7,R+5), (U+0,L+6,D+8,R+2))
_L = cycle_perm(*face_cw_cycles(L), (U+0,F+0,D+0,B+8), (U+3,F+3,D+3,B+5), (U+6,F+6,D+6,B+2))
_R = cycle_perm(*face_cw_cycles(R), (U+2,B+6,D+2,F+2), (U+5,B+3,D+5,F+5), (U+8,B+0,D+8,F+8))

def _inv(p): return invert_perm(p)
def _dbl(p): return compose(p, p)

MOVE_PERMS = {
    "U": _U,  "U'": _inv(_U), "U2": _dbl(_U),
    "D": _D,  "D'": _inv(_D), "D2": _dbl(_D),
    "F": _F,  "F'": _inv(_F), "F2": _dbl(_F),
    "B": _B,  "B'": _inv(_B), "B2": _dbl(_B),
    "L": _L,  "L'": _inv(_L), "L2": _dbl(_L),
    "R": _R,  "R'": _inv(_R), "R2": _dbl(_R),
}

MOVE_NAMES = list(MOVE_PERMS.keys())
MOVE_PERM_LIST = [MOVE_PERMS[m] for m in MOVE_NAMES]

# Inverse move names (needed for backward search)
def _inv_name(m):
    if m.endswith("'"): return m[:-1]
    if m.endswith("2"): return m
    return m + "'"

INV_MOVE_NAME = {m: _inv_name(m) for m in MOVE_NAMES}

def apply_move(cube, move_name):
    perm = MOVE_PERMS[move_name]
    return tuple(cube[perm[i]] for i in range(54))

def apply_move_by_index(state, mv_idx):
    perm = MOVE_PERM_LIST[mv_idx]
    return tuple(state[perm[i]] for i in range(54))

---
## 5. Bidirectional BFS Solver

**Why bidirectional BFS?**

IDA* with a weak heuristic expands O(b^d) nodes where b=18 and d=solution depth.
At depth 10 that's 18^10 ≈ 357 billion nodes: impossible in Python.

Bidirectional BFS expands O(2 × b^(d/2)) nodes for depth 10 that's ~3.5 million.
It searches forward from the scrambled state and backward from the solved state,
stopping when the two frontiers meet.

In [5]:
def bidirectional_bfs(start):
    """
    Bidirectional BFS solver.
    Returns a list of move names to solve `start`, or None if unsolvable.
    Solves any standard scramble (up to ~20 moves) in seconds.
    """
    if start == SOLVED:
        return []

    # Forward search: start -> solved
    # fwd[state] = list of moves from start to reach state
    fwd = {start: []}
    fwd_frontier = deque([start])

    # Backward search: solved -> start (apply inverse moves)
    # bwd[state] = list of moves FROM that state back to solved
    bwd = {SOLVED: []}
    bwd_frontier = deque([SOLVED])

    t0 = time.time()
    depth = 0

    while fwd_frontier or bwd_frontier:
        depth += 1
        print(f"  BFS depth {depth} | fwd={len(fwd)} bwd={len(bwd)} states "
              f"(elapsed {time.time()-t0:.2f}s)", flush=True)

        # --- Expand forward frontier one level ---
        next_fwd = deque()
        while fwd_frontier:
            state = fwd_frontier.popleft()
            path  = fwd[state]
            for mv in MOVE_NAMES:
                nxt = apply_move(state, mv)
                if nxt not in fwd:
                    fwd[nxt] = path + [mv]
                    next_fwd.append(nxt)
                    # Check if backward search has already reached this state
                    if nxt in bwd:
                        return fwd[nxt] + bwd[nxt]
        fwd_frontier = next_fwd

        # --- Expand backward frontier one level ---
        next_bwd = deque()
        while bwd_frontier:
            state = bwd_frontier.popleft()
            path  = bwd[state]
            for mv in MOVE_NAMES:
                # Going backward: apply inverse move
                inv_mv = INV_MOVE_NAME[mv]
                nxt = apply_move(state, inv_mv)
                if nxt not in bwd:
                    # Path from nxt back to solved = [mv] + path
                    bwd[nxt] = [mv] + path
                    next_bwd.append(nxt)
                    # Check if forward search has already reached this state
                    if nxt in fwd:
                        return fwd[nxt] + bwd[nxt]
        bwd_frontier = next_bwd

    return None  # No solution found


# Alias for compatibility
ida_star = bidirectional_bfs
ida_star_fast = bidirectional_bfs

---
## 6. Solvability Check

In [6]:
def check_solvability(cube):
    """Verify each of the 6 colors appears exactly 9 times."""
    counts   = Counter(cube)
    expected = {'W':9,'Y':9,'G':9,'B':9,'O':9,'R':9}
    valid    = True
    print("--- Sticker Count Report ---")
    for color, exp in expected.items():
        got = counts.get(color, 0)
        tag = "OK " if got == exp else "BAD"
        print(f"  {tag} {color}: {got}  (expected {exp})")
        if got != exp:
            valid = False
    print("Status:", "VALID" if valid else "INVALID — fix detection first")
    return valid

---
## 7. Debug Helpers

In [7]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_detection(image_path):
    if not CV2_AVAILABLE:
        print("cv2 not available — cannot visualize detection."); return
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load {image_path}"); return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img_rgb, (300, 300))
    fig, ax = plt.subplots(1)
    ax.imshow(resized)
    cell, m = 100, 15
    for row in range(3):
        for col in range(3):
            rect = patches.Rectangle(
                (col*cell+m, row*cell+m), cell-2*m, cell-2*m,
                linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
    plt.title(f"Detection zones: {image_path}")
    plt.show()


def print_cube(cube):
    names = ['U','D','F','B','L','R']
    for i, name in enumerate(names):
        f = cube[i*9:(i+1)*9]
        print(f"{name}: {f[0]}{f[1]}{f[2]} | {f[3]}{f[4]}{f[5]} | {f[6]}{f[7]}{f[8]}")

---
## 8. Quick Sanity Test (No Images Needed)

In [8]:
def scramble(moves_str):
    """Apply a space-separated sequence of moves to the solved cube."""
    cube = SOLVED
    for mv in moves_str.split():
        cube = apply_move(cube, mv)
    return cube


test_scramble = "R U R' U' F2 L D' B2 R U2"
print(f"Scramble: {test_scramble}")
test_cube = scramble(test_scramble)
print_cube(test_cube)
print()

print("Solving with Bidirectional BFS...")
t0  = time.time()
sol = bidirectional_bfs(test_cube)
dt  = time.time() - t0

if sol:
    print(f"\nSolution ({len(sol)} moves, {dt:.2f}s): {'  '.join(sol)}")
    verify = test_cube
    for mv in sol:
        verify = apply_move(verify, mv)
    print("Verification:", "CORRECT" if is_solved(verify) else "WRONG")
else:
    print("No solution found.")

Scramble: R U R' U' F2 L D' B2 R U2
U: RYR | WWG | YBB
D: WYG | WYG | WWG
F: GRG | YGB | BGY
B: BGB | YBW | RBY
L: YOO | ROO | RBO
R: WOO | RRR | OOW

Solving with Bidirectional BFS...
  BFS depth 1 | fwd=1 bwd=1 states (elapsed 0.00s)
  BFS depth 2 | fwd=19 bwd=19 states (elapsed 0.00s)
  BFS depth 3 | fwd=262 bwd=262 states (elapsed 0.01s)
  BFS depth 4 | fwd=3502 bwd=3502 states (elapsed 0.06s)
  BFS depth 5 | fwd=46733 bwd=46741 states (elapsed 0.88s)

Solution (10 moves, 12.88s): U2  R'  B2  D  L'  F2  U  R  U'  R'
Verification: CORRECT


---
## 9. Full Pipeline: Images → Solution

In [ ]:
print("Reading images...")
current_cube = build_cube_from_images()
print()
print_cube(current_cube)
print()

if not check_solvability(current_cube):
    print("Aborting: fix color detection before solving.")
elif is_solved(current_cube):
    print("The cube is already solved!")
else:
    print("Solving with Bidirectional BFS...")
    t0  = time.time()
    sol = bidirectional_bfs(current_cube)
    dt  = time.time() - t0

    if sol:
        print(f"SOLUTION FOUND in {dt:.2f}s  ({len(sol)} moves)")
        for i, mv in enumerate(sol, 1):
            print(f"  Step {i:>2}: {mv}")
    else:
        print("No solution found — check color detection.")

Reading images...
U.png: ['R', 'G', 'G', 'W', 'W', 'Y', 'O', 'B', 'B']
D.png: ['W', 'W', 'R', 'Y', 'Y', 'B', 'Y', 'Y', 'O']
F.png: ['G', 'O', 'R', 'W', 'B', 'G', 'G', 'G', 'Y']
B.png: ['O', 'R', 'B', 'W', 'G', 'Y', 'W', 'B', 'B']
L.png: ['W', 'O', 'Y', 'R', 'R', 'B', 'O', 'O', 'R']
R.png: ['Y', 'G', 'W', 'O', 'O', 'R', 'G', 'R', 'B']

U: RGG | WWY | OBB
D: WWR | YYB | YYO
F: GOR | WBG | GGY
B: ORB | WGY | WBB
L: WOY | RRB | OOR
R: YGW | OOR | GRB

--- Sticker Count Report ---
  OK  W: 9  (expected 9)
  OK  Y: 9  (expected 9)
  OK  G: 9  (expected 9)
  OK  B: 9  (expected 9)
  OK  O: 9  (expected 9)
  OK  R: 9  (expected 9)
Status: VALID
Solving with Bidirectional BFS...
  BFS depth 1 | fwd=1 bwd=1 states (elapsed 0.00s)
  BFS depth 2 | fwd=19 bwd=19 states (elapsed 0.00s)
  BFS depth 3 | fwd=262 bwd=262 states (elapsed 0.01s)
  BFS depth 4 | fwd=3502 bwd=3502 states (elapsed 0.05s)
  BFS depth 5 | fwd=46733 bwd=46741 states (elapsed 0.63s)
  BFS depth 6 | fwd=622571 bwd=622705 states (